# System Architecture


# Joke Retrieval and Recommendation System (Jester Dataset)

## 1. Problem statement
Given a user (the "query") and a collection of 100 jokes (the "documents"), the system returns a ranked
list of jokes the user is likely to enjoy. It can also answer keyword and Boolean searches over the joke text.

## 2. Dataset
Jester Collaborative Filtering Dataset (Goldberg et al., 2001):
- `JokeText.csv` – 100 jokes (text)
- `UserRatings1.csv`, `UserRatings2.csv` – 73,421 users, 4,136,360 ratings from -10 to +10
- A joke is treated as **relevant** for a user if the user rated it **>= 5**.

## 3. System architecture

```
 JokeText.csv ---> [Text preprocessing] ---> [Inverted index] ---> Boolean search
                   tokenise, stop words,        term -> postings     (AND / OR / NOT)
                   stemming                          |
                                                     v
                                          [TF-IDF vector space] ---> keyword search (cosine)
                                                     |
 UserRatings*.csv ---> [Ratings preprocessing]       |
                       user x joke matrix,           |
                       train / hidden split          |
                            |                        |
                            v                        v
                  +-----------------------------------------------+
                  |  Retrieval models:  Popularity | Content-based |
                  |  Item-based CF | Hybrid (CF + content)        |
                  +-----------------------------------------------+
                                    |
                                    v
                  [Query processing]  user -> score all jokes -> drop seen -> sort -> top-k
                                    |
                                    v
                  [Evaluation]  P@k, Recall@k, MAP, MRR, nDCG@k
```

## 4. Methods used (from the course)
Boolean retrieval and inverted index (Weeks 1-2), TF-IDF and cosine similarity in the vector space
model (Week 3), evaluation metrics (Week 4).

## Step 1: Import libraries and set parameters

In [4]:
pip install numpy pandas scikit-learn matplotlib seaborn

  Using cached numpy-2.5.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.6-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached pillow-12.3.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached numpy-2.5.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Using cached pandas-3.0.6-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 6.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 15.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 18.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 17.5 MB/s  0:00:00
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)
Using cached pillow-12.3.0-cp314-cp314-

In [1]:
pip install nltk

  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.2/807.2 kB 2.8 MB/s  0:00:00 eta 0:00:01
Using cached click-8.5.0-py3-none-any.whl (125 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [nltk]4/5 [nltk]]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.preprocessing import normalize

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

SEED = 42                 # makes the random split reproducible
rng = np.random.default_rng(SEED)

DATA_DIR = "data"            # folder that contains the three CSV files
THRESHOLD = 5.0           # a joke is "relevant" if the user rated it >= 5
TEST_FRACTION = 0.30      # share of each user's ratings hidden for evaluation
N_EVAL_USERS = 3000       # number of users used in the evaluation
MIN_CANDIDATES = 10       # a user needs >= 10 hidden ratings to be evaluated

## Step 2: Load the dataset
The two ratings files hold different users and the same 100 jokes, so they are joined side by side.
We then transpose to a **user x joke** matrix `R`, where `NaN` means "not rated".

In [3]:
jokes = pd.read_csv(f"{DATA_DIR}/JokeText.csv").sort_values("JokeId").reset_index(drop=True)
r1 = pd.read_csv(f"{DATA_DIR}/UserRatings1.csv").set_index("JokeId")
r2 = pd.read_csv(f"{DATA_DIR}/UserRatings2.csv").set_index("JokeId")
assert r1.index.equals(r2.index)
ratings = pd.concat([r1, r2], axis=1).sort_index()   # jokes x users

R = ratings.T.to_numpy(dtype=np.float32)             # users x jokes (NaN = not rated)
rated = np.isfinite(R)
n_users, n_jokes = R.shape

print("Jokes:", n_jokes, "| Users:", n_users, "| Ratings:", f"{rated.sum():,}")
print(f"Missing cells: {1 - rated.mean():.1%} | Rating range: {np.nanmin(R):.2f} to {np.nanmax(R):.2f}")
print(f"Relevant ratings (>= {THRESHOLD}): {(R[rated] >= THRESHOLD).mean():.1%}")

Jokes: 100 | Users: 73421 | Ratings: 4,136,360
Missing cells: 43.7% | Rating range: -9.95 to 10.00
Relevant ratings (>= 5.0): 25.4%


## Step 3: Text preprocessing
Each joke is converted into index terms:
1. **Lower-casing** so "Doctor" and "doctor" match.
2. **Tokenisation**: keep alphabetic words only.
3. **Stop-word removal**: drop very common words (the, and, you...) that carry little meaning.
4. **Stemming** (Porter): reduce words to a root, e.g. "replies" -> "repli", "diseases" -> "diseas".

The same function is used for documents and for queries, so both are matched in the same form.

In [4]:
stemmer = PorterStemmer()
STOP_WORDS = set(ENGLISH_STOP_WORDS) | {"dont", "doesnt", "didnt", "cant", "wont", "im", "ive",
                                        "youre", "thats", "isnt", "wasnt", "hes", "shes"}

def preprocess(text, stem=True, remove_stop=True):
    """Lower-case -> tokenise -> remove stop words -> stem."""
    text = text.lower().replace("'", "")            # don't -> dont
    tokens = re.findall(r"[a-z]+", text)            # keep alphabetic words only
    tokens = [t for t in tokens if len(t) > 1]
    if remove_stop:
        tokens = [t for t in tokens if t not in STOP_WORDS]
    if stem:
        tokens = [stemmer.stem(t) for t in tokens]
    return tokens

example = jokes.loc[0, "JokeText"]
print("ORIGINAL:\n", example)
print("\nTOKENS (no cleaning):", preprocess(example, stem=False, remove_stop=False))
print("\nAFTER STOP-WORD REMOVAL:", preprocess(example, stem=False, remove_stop=True))
print("\nAFTER STEMMING:", preprocess(example))

raw_vocab = {t for txt in jokes.JokeText for t in preprocess(txt, stem=False, remove_stop=False)}
final_vocab = {t for txt in jokes.JokeText for t in preprocess(txt)}
print(f"\nVocabulary: {len(raw_vocab)} raw terms -> {len(final_vocab)} after preprocessing")

ORIGINAL:
 A man visits the doctor. The doctor says "I have bad news for you.You have
cancer and Alzheimer's disease". 
The man replies "Well,thank God I don't have cancer!"


TOKENS (no cleaning): ['man', 'visits', 'the', 'doctor', 'the', 'doctor', 'says', 'have', 'bad', 'news', 'for', 'you', 'you', 'have', 'cancer', 'and', 'alzheimers', 'disease', 'the', 'man', 'replies', 'well', 'thank', 'god', 'dont', 'have', 'cancer']

AFTER STOP-WORD REMOVAL: ['man', 'visits', 'doctor', 'doctor', 'says', 'bad', 'news', 'cancer', 'alzheimers', 'disease', 'man', 'replies', 'thank', 'god', 'cancer']

AFTER STEMMING: ['man', 'visit', 'doctor', 'doctor', 'say', 'bad', 'news', 'cancer', 'alzheim', 'diseas', 'man', 'repli', 'thank', 'god', 'cancer']

Vocabulary: 1548 raw terms -> 1140 after preprocessing


## Step 4: Indexing – inverted index
An **inverted index** maps every term to its *postings list*: the jokes that contain it and how many
times (term frequency). This lets the system find matching jokes without scanning every text.

`term -> {JokeId: term frequency}`

In [5]:
def build_inverted_index(docs):
    index = defaultdict(dict)     # term -> {doc_id: term frequency}
    doc_len = {}
    for doc_id, text in docs.items():
        tokens = preprocess(text)
        doc_len[doc_id] = len(tokens)
        for term, tf in Counter(tokens).items():
            index[term][doc_id] = tf
    return dict(index), doc_len

docs = dict(zip(jokes.JokeId, jokes.JokeText))
inverted_index, doc_len = build_inverted_index(docs)

print("Vocabulary size:", len(inverted_index))
print("Average document length:", round(np.mean(list(doc_len.values())), 1), "terms")
top_terms = sorted(inverted_index, key=lambda t: len(inverted_index[t]), reverse=True)[:8]
print("\nMost frequent terms (document frequency):")
for t in top_terms:
    print(f"  {t:<10} df={len(inverted_index[t])}")
print("\nSample postings list for 'doctor' ->", inverted_index.get(stemmer.stem("doctor")))

Vocabulary size: 1140
Average document length: 26.2 terms

Most frequent terms (document frequency):
  say        df=34
  ask        df=21
  repli      df=20
  man        df=16
  look       df=14
  walk       df=14
  just       df=13
  differ     df=12

Sample postings list for 'doctor' -> {0: 2, 86: 4, 87: 2}


## Step 5: Query processing (1) – Boolean retrieval
Boolean queries combine terms with **AND**, **OR** and **NOT**. They are answered with set operations on
postings lists (intersection, union, complement). Precedence is NOT > AND > OR.
Boolean retrieval returns an unranked set, so it is exact but cannot order the results.

In [6]:
ALL_DOCS = set(docs)

def term_docs(term):
    return set(inverted_index.get(stemmer.stem(term.lower()), {}))

def boolean_search(query):
    """Supports AND, OR, NOT.  Precedence: NOT > AND > OR.  Example: 'doctor AND NOT cancer'."""
    result = set()
    for or_part in query.split(" OR "):
        part_docs = set(ALL_DOCS)
        for term in or_part.split(" AND "):
            term = term.strip()
            if term.startswith("NOT "):
                part_docs &= ALL_DOCS - term_docs(term[4:])
            else:
                part_docs &= term_docs(term)
        result |= part_docs
    return sorted(result)

for q in ["doctor", "doctor AND cancer", "doctor OR lawyer", "doctor AND NOT cancer"]:
    hits = boolean_search(q)
    print(f"{q:<25} -> {len(hits)} jokes {hits[:8]}")

doctor                    -> 3 jokes [0, 86, 87]
doctor AND cancer         -> 1 jokes [0]
doctor OR lawyer          -> 5 jokes [0, 29, 65, 86, 87]
doctor AND NOT cancer     -> 2 jokes [86, 87]


## Step 6: Query processing (2) – TF-IDF vector space model
Each joke becomes a vector of **TF-IDF** weights: a term gets a high weight if it is frequent in the joke
but rare across the collection. Vectors are length-normalised, so the **cosine similarity** between a
query and a joke is a dot product. This gives a *ranked* list of results.

We use sub-linear term frequency (1 + log tf) and our own `preprocess` function as the analyzer.

In [7]:
tfidf = TfidfVectorizer(analyzer=preprocess, sublinear_tf=True)   # uses our own preprocessing
T = tfidf.fit_transform(jokes.JokeText)       # jokes x terms, rows are L2-normalised
T_dense = T.toarray().astype(np.float32)
print("TF-IDF matrix:", T.shape)

def search_jokes(query, k=3):
    """Ranked retrieval of jokes for a free-text query (TF-IDF + cosine similarity)."""
    q = tfidf.transform([query])
    scores = (T @ q.T).toarray().ravel()
    top = np.argsort(-scores)[:k]
    return pd.DataFrame({"rank": range(1, len(top) + 1), "JokeId": jokes.JokeId.values[top],
                         "score": scores[top].round(3),
                         "text": jokes.JokeText.values[top]}).assign(
        text=lambda d: d.text.str.replace(r"\s+", " ", regex=True).str[:90])

print(search_jokes("doctor patient", k=3).to_string(index=False))

TF-IDF matrix: (100, 1140)
 rank  JokeId  score                                                                                       text
    1       0  0.415 A man visits the doctor. The doctor says "I have bad news for you.You have cancer and Alzh
    2      86  0.367 A man, recently completing a routine physical examination receives a phone call from his d
    3      87  0.277 A Czechoslovakian man felt his eyesight was growing steadily worse, and felt it was time t


## Step 7: Ratings preprocessing and train / hidden split
To measure how well the system ranks jokes a user has not been shown, we **hide 30% of each user's
ratings** and build the models only from the remaining 70%.

- Ratings are **mean-centred per user** (a rating of 6 means different things for a generous and a harsh
  rater). Unknown entries are set to 0 after centring.
- The hidden ratings later serve as relevance judgments: hidden rating >= 5 means "relevant".

In [8]:
test_mask = rated & (rng.random(R.shape) < TEST_FRACTION)    # hidden ratings
train_mask = rated & ~test_mask                              # visible ratings

user_mean = np.nanmean(np.where(train_mask, R, np.nan), axis=1)
user_mean = np.nan_to_num(user_mean)
Rc = np.where(train_mask, R - user_mean[:, None], 0).astype(np.float32)   # mean-centred, 0 = unknown
print("Train ratings:", f"{train_mask.sum():,}", "| Hidden ratings:", f"{test_mask.sum():,}")

Train ratings: 2,894,615 | Hidden ratings: 1,241,745


## Step 8: Retrieval model selection
Five rankers are compared; each gives every user a score for every joke.

| Model | Idea | Course link |
|---|---|---|
| Random | Random order (lower bound) | baseline |
| Popularity | Rank jokes by average training rating | baseline |
| Content (TF-IDF) | User profile = TF-IDF vectors of jokes the user liked (weighted by centred rating); rank jokes by cosine similarity to the profile | Vector space model (Week 3) |
| Item-based CF | Score a joke by how similar it is (adjusted cosine on ratings) to jokes the user already rated | Cosine similarity, vector space |
| Hybrid | 0.7 x z-scored CF + 0.3 x z-scored content | Score fusion |

**Why these:** the collection is only 100 jokes but has 73k users, so rating data is rich while text is
thin. We expect collaborative signals to beat text content, and the comparison shows by how much.

In [9]:
item_mean = np.nanmean(np.where(train_mask, R, np.nan), axis=0)

def score_random(users):
    return rng.random((len(users), n_jokes))

def score_popularity(users):
    return np.tile(item_mean, (len(users), 1))

# item-item adjusted cosine similarity from the training ratings
norms = np.sqrt((Rc ** 2).sum(0)) + 1e-9
S = (Rc.T @ Rc) / np.outer(norms, norms)
np.fill_diagonal(S, 0)

def score_item_cf(users):
    num = Rc[users] @ S
    den = train_mask[users].astype(np.float32) @ np.abs(S) + 1e-9
    return num / den

def score_content(users):
    profile = normalize(Rc[users] @ T_dense)   # user profile in TF-IDF space
    return profile @ T_dense.T

def zscore(x):
    return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-9)

def score_hybrid(users, w=0.7):
    return w * zscore(score_item_cf(users)) + (1 - w) * zscore(score_content(users))

MODELS = {"Random": score_random, "Popularity": score_popularity, "Content (TF-IDF)": score_content,
          "Item-based CF": score_item_cf, "Hybrid (CF + content)": score_hybrid}
print("Models:", list(MODELS))

Models: ['Random', 'Popularity', 'Content (TF-IDF)', 'Item-based CF', 'Hybrid (CF + content)']


## Step 9: Query processing (3) – recommending jokes for a user
The query pipeline for a user is:
1. Score all 100 jokes with the chosen model.
2. Remove jokes the user has already rated.
3. Sort by score and return the top-k.

(The keyword and Boolean searches from Steps 5-6 are the text-query side of the system.)

In [10]:
def recommend(user_idx, model="Hybrid (CF + content)", k=5):
    """Query processing: user -> score all jokes -> remove rated jokes -> sort -> top-k."""
    scores = MODELS[model](np.array([user_idx]))[0].copy()
    scores[train_mask[user_idx]] = -np.inf          # do not recommend jokes already seen
    top = np.argsort(-scores)[:k]
    return pd.DataFrame({"rank": range(1, k + 1), "JokeId": jokes.JokeId.values[top],
                         "score": scores[top].round(3),
                         "text": jokes.JokeText.values[top]}).assign(
        text=lambda d: d.text.str.replace(r"\s+", " ", regex=True).str[:80])

demo_user = 123
liked = np.where(train_mask[demo_user] & (R[demo_user] >= THRESHOLD))[0]
print(f"User {demo_user} liked jokes:", liked[:10])
print(recommend(demo_user).to_string(index=False))

User 123 liked jokes: [ 8 24 26 35 97 99]
 rank  JokeId  score                                                                             text
    1      49  0.993 A guy goes into confession and says to the priest, "Father, I'm 80 years old, wi
    2      46  0.903 There was an engineer who had an exceptional gift for fixing all things mechanic
    3      75  0.902 There once was a man and a woman that both got in a terrible car wreck. Both of 
    4      27  0.885 A mechanical, electrical and a software engineer from Microsoft were driving thr
    5      60  0.862 During a recent publicity outing, Hillary sneaked off to visit a fortune teller 


## Step 10: Evaluation metrics
For each user, the **candidates** are the jokes whose ratings were hidden, and every candidate has a known
rating. A candidate is relevant if its rating is >= 5. Each model ranks the candidates and we measure:

- **Precision@k** – share of the top k that are relevant
- **Recall@k** – share of all relevant jokes found in the top k
- **MAP** – mean over users of average precision
- **MRR** – mean reciprocal rank of the first relevant joke
- **nDCG@k** – rank-discounted gain, normalised by the ideal ranking

Only users with at least 10 hidden ratings and at least one relevant hidden joke are evaluated, so
P@10 is always defined. A random sample of 3,000 such users is used.

In [11]:
def precision_at_k(rel, k):  return rel[:k].sum() / k
def recall_at_k(rel, k):     return rel[:k].sum() / rel.sum()
def average_precision(rel):
    hits = np.cumsum(rel)
    return float((hits / np.arange(1, len(rel) + 1) * rel).sum() / rel.sum())
def reciprocal_rank(rel):
    idx = np.flatnonzero(rel)
    return 1.0 / (idx[0] + 1)
def ndcg_at_k(rel, k):
    disc = 1 / np.log2(np.arange(2, k + 2))
    dcg = (rel[:k] * disc[:len(rel[:k])]).sum()
    ideal = (np.sort(rel)[::-1][:k] * disc[:len(rel[:k])]).sum()
    return dcg / ideal

relevant_test = test_mask & (R >= THRESHOLD)
eligible = np.where((test_mask.sum(1) >= MIN_CANDIDATES) & (relevant_test.sum(1) >= 1))[0]
eval_users = rng.choice(eligible, size=min(N_EVAL_USERS, len(eligible)), replace=False)
print(f"Eligible users: {len(eligible):,} | Evaluated: {len(eval_users):,}")
print("Average hidden jokes per user:", round(test_mask[eval_users].sum(1).mean(), 1),
      "| average relevant:", round(relevant_test[eval_users].sum(1).mean(), 1))

Eligible users: 43,763 | Evaluated: 3,000
Average hidden jokes per user: 21.4 | average relevant: 6.5


## Step 11: Run the evaluation
Every model is scored on the same users and the same candidate jokes.

In [12]:
KS = (1, 3, 5, 10)

def evaluate(score_fn):
    scores = score_fn(eval_users)
    rows = []
    for row, u in enumerate(eval_users):
        cand = np.where(test_mask[u])[0]                        # judged candidate jokes
        order = cand[np.argsort(-scores[row, cand], kind="stable")]
        rel = (R[u, order] >= THRESHOLD).astype(float)          # 1 if relevant
        m = {"MAP": average_precision(rel), "MRR": reciprocal_rank(rel)}
        for k in KS:
            m[f"P@{k}"] = precision_at_k(rel, k)
            m[f"R@{k}"] = recall_at_k(rel, k)
            m[f"nDCG@{k}"] = ndcg_at_k(rel, k)
        rows.append(m)
    return pd.DataFrame(rows)

per_user = {name: evaluate(fn) for name, fn in MODELS.items()}
results = pd.DataFrame({name: df.mean() for name, df in per_user.items()}).T
cols = ["P@1", "P@5", "P@10", "R@5", "R@10", "MAP", "MRR", "nDCG@5", "nDCG@10"]
print(results[cols].round(3).to_string())

                         P@1    P@5   P@10    R@5   R@10    MAP    MRR  nDCG@5  nDCG@10
Random                 0.324  0.310  0.310  0.266  0.535  0.408  0.509   0.349    0.437
Popularity             0.496  0.452  0.394  0.421  0.688  0.544  0.654   0.527    0.599
Content (TF-IDF)       0.375  0.362  0.341  0.318  0.595  0.450  0.553   0.407    0.492
Item-based CF          0.546  0.490  0.414  0.455  0.723  0.581  0.697   0.576    0.641
Hybrid (CF + content)  0.528  0.487  0.413  0.449  0.722  0.573  0.685   0.567    0.634
